In [0]:
import json

with open("/Volumes/workspace/job_postings/raw_landings/adzuna_backfill_data_2026-06-22.json", "r") as f:
    data = json.load(f)
print("Top-level keys:", list(data.keys()))
print("Number of results:", len(data["results"]))
print("\nFirst result keys:", list(data["results"][0].keys()))
print("\nFirst result sample:")
print(json.dumps(data["results"][0], indent=2))

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
source_path = f"/Volumes/{catalog}/job_postings/raw_landings/"
checkpoint_path = f"/Volumes/{catalog}/job_postings/raw_landings/_checkpoints/bronze"
schema_path = f"/Volumes/{catalog}/job_postings/raw_landings/_schemas/bronze"
bronze_table = f"{catalog}.job_postings.bronze_job_postings"

# Step 1 — Auto Loader reads each JSON file as a single row
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("multiLine", "true")
    .load(source_path)
)

# Step 2 — Explode results array: one job per row
exploded = (
    raw_stream
    .withColumn("job", F.explode("results"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Step 3 — Flatten nested fields into clean columns
bronze = exploded.select(
    # audit columns
    F.col("_source_file"),
    F.col("_ingested_at"),
    F.col("pulled_at").cast("date").alias("_pulled_at"),
    F.col("run_type").alias("_run_type"),

    # job fields
    F.col("job.id").alias("id"),
    F.col("job.title").alias("title"),
    F.col("job.description").alias("description"),
    F.col("job.created").cast("timestamp").alias("created"),
    F.col("job.contract_type").alias("contract_type"),
    F.col("job.redirect_url").alias("redirect_url"),

    # salary
    F.col("job.salary_min").alias("salary_min"),
    F.col("job.salary_max").alias("salary_max"),
    (F.col("job.salary_is_predicted") == "1").alias("salary_is_predicted"),

    # location — flattened
    F.col("job.location.display_name").alias("location_display"),
    F.col("job.location.area").alias("location_area"),

    # company and category
    F.col("job.company.display_name").alias("company"),
    F.col("job.category.label").alias("category_label"),
)

# Step 4 — Write to Bronze Delta table
(
    bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)